# Day 1: Environment Setup & Data Exploration

Goal: Load PubMedQA and run one forward pass of Mistral-7B.

In [ ]:
!pip install -q datasets transformers peft trl bitsandbytes accelerate wandb evaluate rouge_score -U

In [ ]:
from datasets import load_dataset

# Load labeled split first (1000 samples, fast)
ds_labeled = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
print(f"Labeled samples: {len(ds_labeled)}")
print(f"Columns: {ds_labeled.column_names}")
print(f"Example: {ds_labeled[0]}")

In [ ]:
# Load artificial split (211k samples, use streaming for exploration)
ds_artificial = load_dataset("qiaojin/PubMedQA", "pqa_artificial", split="train")
print(f"Artificial samples: {len(ds_artificial)}")

In [ ]:
# Label distribution
from collections import Counter
labels = Counter(ds_labeled["final_decision"])
print(f"Label distribution (labeled): {labels}")
labels_art = Counter(ds_artificial["final_decision"])
print(f"Label distribution (artificial): {labels_art}")

In [ ]:
# Token length distribution
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")

def get_token_length(sample):
    abstract = " ".join(sample["context"]["contexts"])
    text = f"Context: {abstract}\nQuestion: {sample['question']}\nAnswer: {sample['final_decision']}"
    return {"token_count": len(tokenizer.encode(text))}

sample_lengths = ds_labeled.map(get_token_length)
lengths = sample_lengths["token_count"]
print(f"Mean: {sum(lengths)/len(lengths):.0f}, Max: {max(lengths)}, P95: {sorted(lengths)[int(0.95*len(lengths))]}")